# 👩‍💻 Building and Evaluating a StackingClassifier on Loan Default Data

## 📋 Overview
In this activity, you will put your machine learning skills to the test by creating a stacked ensemble model to predict loan defaults using a dataset from Lending Club. Stacking allows you to leverage multiple algorithms to improve predictive accuracy, an essential skill in a world where financial institutions rely on robust models to manage risk and maximize profitability. By the end of this lab, you will be proficient in constructing and assessing a stacking model using the `StackingClassifier`.

## 🎯 Learning Outcomes
By the end of this lab, you will be able to:

- ✅ Construct a stacked ensemble model using multiple base models and a meta-model
- ✅ Evaluate the performance of the stacking model using various classification metrics
- ✅ Reflect on model selection and explore techniques for improving predictive performance

## Task 1: Data Exploration and Preparation

**Context:** Understanding the dataset and ensuring it is clean is the first critical step.

**Steps:**

1. Load the Lending Club Loan Dataset from the provided CSV file.
2. Conduct exploratory data analysis (EDA) including:
    - Displaying summary statistics
    - Checking for missing values
    - Identifying categorical features

In [1]:
# Task 1: Data Exploration and Preparation
# Required Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Load Data
df = pd.read_csv('lending_club_loan_data.csv')

# EDA
print(f"Shape of Data: {df.shape}")

print("------- Top 5 Rows of Data -------")
display(df.head())
print()

print("------- Information of Data -------")
print(df.info())
print()

print("------- Summary of Data -------")
display(df.describe().T)
print()

print("------- Data Types of Data -------")
print(df.dtypes)

Shape of Data: (39202, 41)
------- Top 5 Rows of Data -------


,loan_amnt,int_rate,installment,emp_length,annual_inc,loan_status,zip_code,dti,delinq_2yrs,fico_range_high,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,term_ 36 months,term_ 60 months
0,5000.0,10.65,162.87,10,24000.0,1,860,27.65,0.0,739.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2500.0,15.27,59.83,0,30000.0,0,309,1.00,0.0,744.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2400.0,15.96,84.33,10,12252.0,1,606,8.72,0.0,739.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,10000.0,13.49,339.31,10,49200.0,1,917,20.00,0.0,694.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,5000.0,7.90,156.46,3,36000.0,1,852,11.20,0.0,734.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0



------- Information of Data -------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39202 entries, 0 to 39201
Data columns (total 41 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   loan_amnt                            39202 non-null  float64
 1   int_rate                             39202 non-null  float64
 2   installment                          39202 non-null  float64
 3   emp_length                           39202 non-null  int64  
 4   annual_inc                           39202 non-null  float64
 5   loan_status                          39202 non-null  int64  
 6   zip_code                             39202 non-null  int64  
 7   dti                                  39202 non-null  float64
 8   delinq_2yrs                          39202 non-null  float64
 9   fico_range_high                      39202 non-null  float64
 10  inq_last_6mths                       39202 non-null  floa

,count,mean,std,min,25%,50%,75%,max
loan_amnt,39202.0,11145.393092,7400.190394,500.00,5425.00,10000.00,15000.0000,35000.00
int_rate,39202.0,11.978145,3.707822,5.42,8.94,11.83,14.4600,24.59
installment,39202.0,323.521056,208.481659,15.69,166.50,279.16,427.9775,1305.19
emp_length,39202.0,4.828606,3.603876,0.00,2.00,4.00,9.0000,10.00
annual_inc,39202.0,68919.956784,63990.170524,4000.00,40206.00,59000.00,82000.0000,6000000.00
loan_status,39202.0,0.855875,0.351221,0.00,1.00,1.00,1.0000,1.00
zip_code,39202.0,555.055150,302.273987,100.00,278.00,600.00,853.0000,999.00
dti,39202.0,13.298647,6.675006,0.00,8.16,13.39,18.5800,29.99
delinq_2yrs,39202.0,0.146600,0.491464,0.00,0.00,0.00,0.0000,11.00
fico_range_high,39202.0,719.015560,35.874179,629.00,689.00,714.00,744.0000,829.00



------- Data Types of Data -------
loan_amnt                              float64
int_rate                               float64
installment                            float64
emp_length                               int64
annual_inc                             float64
loan_status                              int64
zip_code                                 int64
dti                                    float64
delinq_2yrs                            float64
fico_range_high                        float64
inq_last_6mths                         float64
open_acc                               float64
pub_rec                                float64
revol_bal                              float64
revol_util                             float64
total_acc                              float64
last_fico_range_high                   float64
home_ownership_MORTGAGE                float64
home_ownership_NONE                    float64
home_ownership_OTHER                   float64
home_ownership_OWN      

In [2]:
scaler = StandardScaler()
label_encoder = LabelEncoder()
for column in df.select_dtypes(include=['object']).columns:
    df[column] = label_encoder.fit_transform(df[column])

X = df.drop('loan_status', axis=1)
y = df['loan_status']
X = scaler.fit_transform(X)

💡 **Tip:** Use `pd.read_csv()` to load the data, and `df.describe()` to get summary statistics.

⚙️ **Test Your Work:**

- Display the first 5 rows of the dataset.

**Expected output:** A preview of the data with columns such as 'loan_amnt', 'term', 'int_rate', etc.

## Task 2: Split the Data

**Context:** Splitting the data allows for an unbiased evaluation of the model.

**Steps:**

1. Divide the dataset into training and testing sets with an 80-20 split.

In [3]:
# Task 2: Split the Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Shape of X Train: {X_train.shape}")
print(f"Shape of Y Train: {y_train.shape}")
print(f"Shape of X Test: {X_test.shape}")
print(f"Shape of Y Test: {y_test.shape}")

Shape of X Train: (31361, 40)
Shape of Y Train: (31361,)
Shape of X Test: (7841, 40)
Shape of Y Test: (7841,)


💡 **Tip:** Use `train_test_split` with a `test_size` of 0.2 and a `random_state` for reproducibility.

⚙️ **Test Your Work:**

- Print the shapes of the training and testing sets.

**Expected output:** Shapes that reflect the 80-20 split.

## Task 3: Define Base Models

**Context:** Base models capture different patterns within the dataset, enhancing the final model's performance.

**Steps:**

1. Choose a set of diverse base models such as Decision Tree, Support Vector Machine, and Logistic Regression.

In [4]:
# Task 3: Define Base Models
base_models = [
    ('decision_tree', DecisionTreeClassifier(max_depth=3)),
    ('random_forest', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('support_vector_machine', SVC(probability=True))
]

💡 **Tip:** Ensure each model is instantiated with appropriate parameters.

⚙️ **Test Your Work:**

- Print the base models' configurations.

**Expected output:** Configurations of the base models being used in the stack.

## Task 4: Build the Stacking Model

**Context:** Combining base models and a meta-model leads to a more robust predictive model.

**Steps:**
1. Construct a `StackingClassifier` using the defined base models.
2. Select a meta-model, typically a simpler model such as Logistic Regression.

In [5]:
# Task 4: Build the Stacking Model

meta_model = LogisticRegression()
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model)

💡 **Tip:** Use `estimators` parameter for base models and `final_estimator` for the meta-model.

⚙️ **Test Your Work:**

- Print the `StackingClassifier` configuration.

**Expected output:** The configuration detailing the base models and the meta-model.

## Task 5: Train and Evaluate the Model

**Context:** Fitting the model to the training data and evaluating its performance is essential.

**Steps:**

1. Fit the StackingClassifier to the training data using fit.
2. Evaluate the model using metrics such as accuracy, precision, recall, and F1 score.

In [6]:
# Task 5: Train and Evaluate the Model

stacking_clf.fit(X_train, y_train)

y_pred = stacking_clf.predict(X_test)
print(f"Stacking Model Evaluation:\n {classification_report(y_test, y_pred)}")

Stacking Model Evaluation:
               precision    recall  f1-score   support

           0       0.66      0.44      0.53      1170
           1       0.91      0.96      0.93      6671

    accuracy                           0.88      7841
   macro avg       0.78      0.70      0.73      7841
weighted avg       0.87      0.88      0.87      7841



💡 **Tip:** Use `classification_report` to get detailed performance metrics.

⚙️ **Test Your Work:**

- Print the classification report for the stacking model’s predictions.

**Expected output:** Metrics including accuracy, precision, recall, and F1 score.

### ✅ Success Checklist

- Successfully explored and prepared the dataset
- Split the data into training and testing sets
- Defined and configured diverse base models
- Constructed and configured the stacking model
- Trained and evaluated the stacking model
- Reflected on the model selection and potential improvements

### 🔍 Common Issues & Solutions

**Problem:** Data leakage during preprocessing.Dataset file not found.   
**Solution:** Ensure the dataset file is in the correct folder.

**Problem:** Categorical encoding errors.   
**Solution:** Double-check the columns being encoded.

**Problem:** Model training errors.   
**Solution:** Verify that data preprocessing steps were correctly applied.

### 🔑 Key Points

- Stacking allows leveraging multiple algorithms for better predictive performance.
- Proper data preprocessing is essential for model accuracy.
- Evaluating model metrics helps understand and improve model performance.

## 💻 Exemplar Solution

<details>    
<summary><strong>Click HERE to see an exemplar solution</strong></summary>    

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Load Data
df = pd.read_csv('lending_club_loan_data.csv')

# Data preprocessing
print(df.describe())
print(df.dtypes)

# Standardizing data
scaler = StandardScaler()
label_encoder = LabelEncoder()
for column in df.select_dtypes(include=['object']).columns:
    df[column] = label_encoder.fit_transform(df[column])

X = df.drop('loan_status', axis=1)
y = df['loan_status']
X = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define base models
base_models = [
    ('decision_tree', DecisionTreeClassifier(max_depth=3)),
    ('random_forest', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('support_vector_machine', SVC(probability=True))
]

# Stacking model
meta_model = LogisticRegression()
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model)

# Train the stacking classifier
stacking_clf.fit(X_train, y_train)

# Evaluate
y_pred = stacking_clf.predict(X_test)
print(f"Stacking Model Evaluation:\n {classification_report(y_test, y_pred)}")
```  